In [0]:
CREATE OR REPLACE TABLE default.olist_orders_cleaned AS
SELECT 
  order_id,
  customer_id,
  
  -- Standardize order_status text to lowercase
  LOWER(order_status) AS order_status, 
  
  order_purchase_timestamp,
  order_approved_at,
  
  -- Keep missing carrier shipment date as NULL
  order_delivered_carrier_date,
  
  -- Keep missing customer delivery date as NULL
  order_delivered_customer_date,
  
  order_estimated_delivery_date,
  
  -- New column: is_delivered flag
  CASE 
    WHEN order_delivered_customer_date IS NULL THEN 'No'
    ELSE 'Yes'
  END AS is_delivered,
  
  -- New column: is_late_delivery flag
  CASE 
    WHEN order_delivered_customer_date IS NULL THEN NULL
    WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Yes'
    ELSE 'No'
  END AS is_late_delivery
  
FROM default.olist_orders_dataset;


num_affected_rows,num_inserted_rows


In [0]:
SELECT 
  DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS order_month,
  ROUND(SUM(CASE WHEN is_late_delivery = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS late_rate_percentage
FROM default.olist_orders_cleaned
WHERE is_delivered = 'Yes'
GROUP BY order_month
ORDER BY late_rate_percentage DESC
LIMIT 5;


order_month,late_rate_percentage
2016-09,100.00
2018-03,21.36
2018-02,16.00
2017-11,14.31
2018-08,10.39


In [0]:
SELECT 
  DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS order_month,
  ROUND(SUM(CASE WHEN is_late_delivery = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS late_rate_percentage
FROM default.olist_orders_cleaned
WHERE is_delivered = 'Yes'
GROUP BY order_month
ORDER BY order_month;


order_month,late_rate_percentage
2016-09,100.00
2016-10,1.11
2016-12,0.00
2017-01,3.07
2017-02,3.21
2017-03,5.58
2017-04,7.86
2017-05,3.61
2017-06,3.86
2017-07,3.43


In [0]:
SELECT 
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM default.olist_orders_cleaned), 2) AS canceled_percentage
FROM default.olist_orders_cleaned
WHERE order_status = 'canceled';


canceled_percentage
0.63


In [0]:
SELECT 
  is_delivered, 
  COUNT(*) AS total_orders,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM default.olist_orders_cleaned), 2) AS percentage
FROM default.olist_orders_cleaned
GROUP BY is_delivered
ORDER BY total_orders DESC;


is_delivered,total_orders,percentage
Yes,96476,97.02
No,2965,2.98


In [0]:

SELECT 
  is_late_delivery, 
  COUNT(*) AS total_orders,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM default.olist_orders_cleaned WHERE is_delivered = 'Yes'), 2) AS percentage
FROM default.olist_orders_cleaned
WHERE is_delivered = 'Yes'
GROUP BY is_late_delivery
ORDER BY total_orders DESC;


is_late_delivery,total_orders,percentage
No,88649,91.89
Yes,7827,8.11


In [0]:
SELECT 
  order_status,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN is_delivered = 'Yes' THEN 1 ELSE 0 END) AS delivered_orders,
  SUM(CASE WHEN is_delivered = 'No' THEN 1 ELSE 0 END) AS not_delivered_orders
FROM default.olist_orders_cleaned
GROUP BY order_status
ORDER BY total_orders DESC;


order_status,total_orders,delivered_orders,not_delivered_orders
delivered,96478,96470,8
shipped,1107,0,1107
canceled,625,6,619
unavailable,609,0,609
invoiced,314,0,314
processing,301,0,301
created,5,0,5
approved,2,0,2


In [0]:
SELECT 
  ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)), 2) AS avg_delivery_days
FROM default.olist_orders_cleaned
WHERE is_delivered = 'Yes' AND order_delivered_customer_date IS NOT NULL;

avg_delivery_days
12.5
